<a href="https://colab.research.google.com/github/Developer-Abnam/machine-learning-zoomcamp-2025-solutions/blob/main/week_3_machine_learning_for_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 3rd Week's Homework by - Abdulsamad Nuradeen


In [ ]:
data = 'https://raw.githubusercontent.com/alexeygrigorev/datasets/master/course_lead_scoring.csv'

In [ ]:
!wget $data

--2025-10-12 08:37:21--  https://raw.githubusercontent.com/alexeygrigorev/datasets/master/course_lead_scoring.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.108.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 80876 (79K) [text/plain]
Saving to: ‘course_lead_scoring.csv.1’

course_lead_scoring 100%[===================>]  78.98K  --.-KB/s    in 0.01s   

2025-10-12 08:37:22 (5.83 MB/s) - ‘course_lead_scoring.csv.1’ saved [80876/80876]



In [ ]:
import pandas as pd
import numpy as np

**Data Preparation**

In [ ]:
df = pd.read_csv(data)
df.head()

,lead_source,industry,number_of_courses_viewed,annual_income,employment_status,location,interaction_count,lead_score,converted
0,paid_ads,NaN,1,79450.0,unemployed,south_america,4,0.94,1
1,social_media,retail,1,46992.0,employed,south_america,1,0.80,0
2,events,healthcare,5,78796.0,unemployed,australia,3,0.69,1
3,paid_ads,retail,2,83843.0,NaN,australia,1,0.87,0
4,referral,education,3,85012.0,self_employed,europe,3,0.62,1


**Data preparation**



*   Check if the missing values are presented in the features.
*   If there are missing values:
*   For caterogiral features, replace them with 'NA'
*    For numerical features, replace with with 0.0





    
   

In [ ]:
df.isna().sum()

,0
lead_source,128
industry,134
number_of_courses_viewed,0
annual_income,181
employment_status,100
location,63
interaction_count,0
lead_score,0
converted,0


In [ ]:
df_filled = df.copy()
# categorical columns
categorical_cols = df_filled.select_dtypes('object').columns.tolist()

# numerical columns
numerical_cols = df_filled.select_dtypes(exclude='object').columns.tolist()

# For categorical features: replace NaN with 'NA'
if categorical_cols:
    df_filled[categorical_cols] = df_filled[categorical_cols].fillna('NA')
    print("Filled missing values in categorical columns with 'NA'")

# For numerical features: replace NaN with 0.0
if numerical_cols:
    df_filled[numerical_cols] = df_filled[numerical_cols].fillna(0.0)
    print("Filled missing values in numerical columns with 0.0")

Filled missing values in categorical columns with 'NA'
Filled missing values in numerical columns with 0.0


In [ ]:
df_filled.isna().sum()

,0
lead_source,0
industry,0
number_of_courses_viewed,0
annual_income,0
employment_status,0
location,0
interaction_count,0
lead_score,0
converted,0


# Question 1:

What is the most frequent observation (mode) for the column industry?

*   NA
*   Technology
*   healthcare
*   retail

In [ ]:
df_filled['industry'].mode()[0]

'retail'

# Question 2

Create the correlation matrix for the numerical features of your dataset. In a correlation matrix, you compute the correlation coefficient between every pair of features.

What are the two features that have the biggest correlation?

*   interaction_count and lead_score
*   number_of_courses_viewed and lead_score
*   number_of_courses_viewed and interaction_count
*   annual_income and interaction_count





Only consider the pairs above when answering this question.

In [ ]:
corr = df_filled.corr(numeric_only=True).unstack().reset_index().rename(columns={0:"corr"})
corr = corr[corr['corr'] != 1].copy()
corr

,level_0,level_1,corr
1,number_of_courses_viewed,annual_income,0.009770
2,number_of_courses_viewed,interaction_count,-0.023565
3,number_of_courses_viewed,lead_score,-0.004879
4,number_of_courses_viewed,converted,0.435914
5,annual_income,number_of_courses_viewed,0.009770
7,annual_income,interaction_count,0.027036
8,annual_income,lead_score,0.015610
9,annual_income,converted,0.053131
10,interaction_count,number_of_courses_viewed,-0.023565
11,interaction_count,annual_income,0.027036


**Split the data**

*   Split your data in train/val/test sets with 60%/20%/20% distribution.
*   Use Scikit-Learn for that (the train_test_split function) and set the seed to 42.
*   Make sure that the target value y is not in your dataframe.




In [ ]:
from sklearn.model_selection import train_test_split
seed = 42
df_X, df_y = df_filled.drop('converted', axis=1).copy(), df_filled['converted'].copy()
X_train, X_val, y_train, y_val = train_test_split(df_X, df_y, test_size=0.6, random_state=seed)
X_val, X_test, y_val, y_test = train_test_split(X_val, y_val, test_size=0.5, random_state=seed)

# Question 3

*   Calculate the mutual information score between y and other categorical variables in the dataset. Use the training set only.
*   Round the scores to 2 decimals using round(score, 2).






In [ ]:
from sklearn.metrics import mutual_info_score
def MI_score(X,y):
  return mutual_info_score(X, y_train)
X_train.select_dtypes('object').apply(lambda x: round(MI_score(x, y_train), 2)).sort_index(ascending=False).to_frame(name='MI')

,MI
location,0.01
lead_source,0.03
industry,0.02
employment_status,0.01


# One Hot Encoding

In [ ]:

from sklearn.feature_extraction import DictVectorizer
train_dicts = X_train.to_dict(orient='records')
val_dicts = X_val.to_dict(orient='records')
test_dicts = X_test.to_dict(orient='records')

dv = DictVectorizer(sparse=False).set_output(transform='pandas').fit(train_dicts)

X_train = dv.transform(train_dicts)
X_val = dv.transform(val_dicts)
X_test = dv.transform(test_dicts)

X_train.head()

,annual_income,employment_status=NA,employment_status=employed,employment_status=self_employed,employment_status=student,employment_status=unemployed,industry=NA,industry=education,industry=finance,industry=healthcare,...,lead_source=social_media,location=NA,location=africa,location=asia,location=australia,location=europe,location=middle_east,location=north_america,location=south_america,number_of_courses_viewed
0,31353.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1,69518.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0
2,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,4.0
3,52131.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,2.0
4,58861.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0


# Question 4

*   Now let's train a logistic regression.
*   Remember that we have several categorical variables in the dataset. Include them using one-hot encoding.
*   Fit the model on the training dataset.
*   To make sure the results are reproducible across different versions of Scikit-Learn, fit the model with these parameters:

  model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)

  Calculate the accuracy on the validation dataset and round it to 2 decimal digits.
  
  What accuracy did you get?

In [ ]:
from sklearn.linear_model import LogisticRegression
model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model.fit(X_train, y_train)

LogisticRegression(max_iter=1000, random_state=42, solver='liblinear')

In [ ]:
y_pred_val = model.predict_proba(X_val)[:, 1] >= 0.5
accuracy = round((y_val == y_pred_val).mean(), 2)
accuracy

np.float64(0.74)

# Question 5


*   Let's find the least useful feature using the feature elimination technique.
*   Train a model using the same features and parameters as in Q4 (without rounding).
*   Now exclude each feature from this set and train a model without it. Record the accuracy for each model.
*   For each feature, calculate the difference between the original accuracy and the accuracy without the feature.

Which of following feature has the smallest difference?
*   'industry'
*   'employment_status'
*   'lead_score'













In [ ]:
feats_list = ['industry', 'employment_status', 'lead_score']
accuracy_features = []
for feat in feats_list:
  from sklearn.linear_model import LogisticRegression
  model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
  mask = ~X_train.columns.str.contains(feat)
  model.fit(X_train.loc[:, mask], y_train)
  y_pred_val = model.predict_proba(X_val.loc[:, mask])[:, 1] >= 0.5
  accuracy = (y_val == y_pred_val).mean()
  accuracy_features.append(accuracy)

pd.DataFrame(data={'accuracy': accuracy-accuracy_features}, index=[f'without_{feat}' for feat in feats_list]).sort_values(by='accuracy', ascending=False)


,accuracy
without_employment_status,0.004556
without_lead_score,0.000000
without_industry,-0.002278


# Question 6
+ Now let's train a regularized logistic regression.
+ Let's try the following values of the parameter `C`: `[0.01, 0.1, 1, 10, 100]`.
+ Train models using all the features as in Q4.
+ Calculate the accuracy on the validation dataset and round it to 3 decimal digits.

Which of these `C` leads to the best accuracy on the validation set?

+ 0.01
+ 0.1
+ 1
+ 10
+ 100

>Note: If there are multiple options, select the smallest C.

In [ ]:
C_list = [0.01, 0.1, 1, 10, 100]
accuracy_C = []
for c in C_list:
  model = LogisticRegression(solver='liblinear', C=c, max_iter=1000, random_state=42)
  model.fit(X_train, y_train)
  y_pred_val = model.predict_proba(X_val)[:, 1] >= 0.5
  accuracy = round((y_val == y_pred_val).mean(), 3)
  accuracy_C.append(accuracy)


pd.DataFrame(data={'C':C_list, 'accuracy':accuracy_C}).sort_values(by='accuracy', ascending=False)

,C,accuracy
1,0.10,0.743
3,10.00,0.743
2,1.00,0.743
4,100.00,0.743
0,0.01,0.736
